# Day 16 — LangChain Foundations
## 30 Days of AI: From NLP to LLMs

---

Days 11–15 you built RAG from scratch: every component written
by hand — chunker, vector store, retrieval, prompt assembly,
generation. You now understand every layer deeply.

LangChain is the most widely used framework for building LLM
applications. It provides pre-built, composable abstractions
for the exact patterns you built manually. The reason to learn
it now — after building from scratch — is that you understand
what it is doing under the hood. You are not just calling magic.

LangChain is most useful for chaining multiple LLM calls,
managing memory in conversations, connecting to external tools,
and building agents. These are patterns that become tedious
to maintain manually as complexity grows.

---

### What You Will Learn Today

- LangChain architecture — chains, models, prompts, memory, tools
- LLM and ChatModel wrappers — unified interface across providers
- PromptTemplate and ChatPromptTemplate — reusable prompt factories
- LLMChain — the fundamental building block
- SequentialChain — pipe output of one chain into the next
- Memory — ConversationBufferMemory, ConversationSummaryMemory
- LangChain RAG — using built-in document loaders and retrievers
- LCEL — LangChain Expression Language for composing pipelines

### Goal by End of Day

Build a multi-step LangChain pipeline that processes a document,
summarizes it, classifies it, and answers questions about it —
all chained together. Rebuild your Day 15 RAG using LangChain
components and compare the code length.

In [ ]:
## Run once
## !pip install langchain langchain-openai langchain-anthropic \
##             langchain-community sentence-transformers faiss-cpu -q

import os
import warnings
warnings.filterwarnings('ignore')

# Check which API key is available
HAS_OPENAI    = bool(os.environ.get('OPENAI_API_KEY'))
HAS_ANTHROPIC = bool(os.environ.get('ANTHROPIC_API_KEY'))

print('API Keys detected:')
print(f'  OpenAI    : {"yes" if HAS_OPENAI    else "no"}')
print(f'  Anthropic : {"yes" if HAS_ANTHROPIC else "no"}')
print()

if not HAS_OPENAI and not HAS_ANTHROPIC:
    print('No API key found.')
    print('Set one with:')
    print('  export OPENAI_API_KEY=sk-...')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('This notebook includes mock outputs for all cells so you')
    print('can read through without a key. Set a key for live results.')

print('LangChain version check:')
try:
    import langchain
    print(f'  langchain : {langchain.__version__}')
except ImportError:
    print('  langchain not installed — run the pip install above')

---

## Part 1 — LangChain Architecture Overview

LangChain organizes LLM application code into six abstractions:

```
┌─────────────────────────────────────────────────────────────────┐
│  MODELS                                                         │
│  Unified interface to any LLM provider (OpenAI, Anthropic,     │
│  Cohere, local HuggingFace models, etc.)                       │
│  ChatOpenAI, ChatAnthropic, HuggingFacePipeline                │
├─────────────────────────────────────────────────────────────────┤
│  PROMPTS                                                        │
│  PromptTemplate — parameterized prompt strings                 │
│  ChatPromptTemplate — system + user message templates          │
│  FewShotPromptTemplate — automatic few-shot example injection  │
├─────────────────────────────────────────────────────────────────┤
│  CHAINS                                                         │
│  LLMChain — prompt + model in one call                         │
│  SequentialChain — pipe outputs between chains                 │
│  RetrievalQA — RAG: retriever + prompt + model                 │
├─────────────────────────────────────────────────────────────────┤
│  MEMORY                                                         │
│  ConversationBufferMemory — store full history                 │
│  ConversationSummaryMemory — summarize old turns               │
│  ConversationTokenBufferMemory — trim at token limit           │
├─────────────────────────────────────────────────────────────────┤
│  RETRIEVERS & VECTOR STORES                                     │
│  FAISS, Chroma, Pinecone — same stores you built on Day 14     │
│  MultiQueryRetriever, SelfQueryRetriever, EnsembleRetriever    │
├─────────────────────────────────────────────────────────────────┤
│  AGENTS & TOOLS  (Day 17)                                       │
│  Tool — wrap any function as an LLM-callable tool              │
│  AgentExecutor — ReAct loop: think → act → observe → repeat   │
└─────────────────────────────────────────────────────────────────┘

LCEL (LangChain Expression Language):
  The modern way to compose chains using the | pipe operator.
  prompt | model | output_parser
  Lazy evaluation, streaming support, automatic tracing.
```

In [ ]:
# ----------------------------------------------------------------
# Part 2 — Models: Unified LLM interface
# ----------------------------------------------------------------

# LangChain wraps every provider behind the same interface.
# .invoke() replaces raw API calls. Same code works with any model.

def get_llm(temperature=0.3):
    """Return a LangChain LLM based on available API keys."""
    if HAS_OPENAI:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model       = 'gpt-3.5-turbo',
            temperature = temperature,
        )
    elif HAS_ANTHROPIC:
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(
            model       = 'claude-3-haiku-20240307',
            temperature = temperature,
        )
    else:
        # Fake LLM for demonstration — returns mock responses
        from langchain_core.language_models.fake import FakeListChatModel
        return FakeListChatModel(responses=[
            'This is a mock LLM response. Set an API key for real output.',
            'Mock response 2: The answer is grounded in the provided context.',
            'Mock response 3: Positive sentiment detected.',
            'Mock response 4: Summary complete.',
        ] * 20)


llm = get_llm()
print(f'LLM loaded: {type(llm).__name__}')
print()

# Simplest possible call: .invoke() takes a string or messages
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content='You are a concise AI tutor. Answer in 1 sentence.'),
    HumanMessage(content='What is a transformer in machine learning?'),
]

response = llm.invoke(messages)

print('Direct LLM call via .invoke()')
print('=' * 55)
print('Response:', response.content)
print('Type    :', type(response).__name__)
if hasattr(response, 'usage_metadata'):
    print('Tokens  :', response.usage_metadata)

In [ ]:
# ----------------------------------------------------------------
# Part 3 — PromptTemplate: reusable, parameterized prompts
# ----------------------------------------------------------------

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# --- Simple PromptTemplate ---
# {variable} placeholders are filled at call time

summary_template = PromptTemplate(
    input_variables = ['text', 'audience', 'sentences'],
    template = (
        'Summarize the following text in exactly {sentences} sentences.\n'
        'Audience: {audience}. Be concise and factual.\n\n'
        'Text:\n{text}\n\nSummary:'
    )
)

# Format the prompt without calling the LLM yet
filled_prompt = summary_template.format(
    text      = 'Machine learning enables computers to learn from data...',
    audience  = 'non-technical manager',
    sentences = 2,
)
print('PromptTemplate.format() output:')
print(filled_prompt)
print()

# --- ChatPromptTemplate --- 
# For system + user message structure

qa_chat_template = ChatPromptTemplate.from_messages([
    ('system', (
        'You are an expert {domain} tutor. '
        'Answer clearly in {style} style.'
    )),
    ('human', '{question}'),
])

messages = qa_chat_template.format_messages(
    domain   = 'machine learning',
    style    = 'concise bullet-point',
    question = 'What are the main types of machine learning?',
)

print('ChatPromptTemplate.format_messages() output:')
for msg in messages:
    print(f'  [{msg.type.upper()}] {msg.content[:80]}')

In [ ]:
# ----------------------------------------------------------------
# Part 4 — LCEL: LangChain Expression Language
# The modern way to build chains using | (pipe operator)
# ----------------------------------------------------------------

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser extracts .content from the AIMessage
# so you get a plain string back

# Build a chain: prompt | llm | parser
# Each | passes output of left as input to right

explain_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are an expert teacher. Explain concepts simply.'),
        ('human',  'Explain {concept} in 2 sentences for a {audience}.'),
    ])
    | get_llm(temperature=0.3)
    | StrOutputParser()
)

# .invoke() runs the full chain
result = explain_chain.invoke({
    'concept'  : 'gradient descent',
    'audience' : 'high school student',
})

print('LCEL Chain: prompt | llm | StrOutputParser')
print('=' * 55)
print('Output:', result)
print()

# .batch() runs on multiple inputs in parallel
concepts = [
    {'concept': 'overfitting',     'audience': 'business manager'},
    {'concept': 'neural network',  'audience': 'biology student'},
    {'concept': 'attention heads', 'audience': 'software engineer'},
]

batch_results = explain_chain.batch(concepts)

print('Batch execution (3 inputs):')
print('=' * 55)
for inp, out in zip(concepts, batch_results):
    print(f'Concept [{inp["concept"]}] for [{inp["audience"]}]:')
    print(f'  {out[:120]}...')
    print()

In [ ]:
# ----------------------------------------------------------------
# Part 5 — Sequential Chains: pipe output into next chain
# Classic pattern: document → summarize → classify → respond
# ----------------------------------------------------------------

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser

DOCUMENT = """
Researchers at Stanford have developed a new protein folding algorithm
that outperforms AlphaFold2 on 78% of benchmark proteins. The method,
called ProFold-X, uses a novel attention mechanism that models both
local and global structural constraints simultaneously. Training took
3 weeks on 512 A100 GPUs using a dataset of 200 million protein sequences.
The team published their findings in Nature Biotechnology and has made
the weights publicly available on HuggingFace.
"""

llm_instance = get_llm(temperature=0.2)

# Chain 1: Summarize
summarize_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'Summarize research papers in 1 sentence.'),
        ('human',  'Summarize: {document}'),
    ])
    | llm_instance
    | StrOutputParser()
)

# Chain 2: Extract key facts from the summary
extract_chain = (
    ChatPromptTemplate.from_messages([
        ('system',
         'Extract key facts from text. Return ONLY valid JSON with keys: '
         'topic (str), institution (str), key_metric (str), status (str).'),
        ('human', '{summary}'),
    ])
    | llm_instance
    | StrOutputParser()
)

# Chain 3: Generate a tweet from the summary
tweet_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'Write engaging science tweets. Max 280 chars. Use 1-2 emojis.'),
        ('human',  'Write a tweet about: {summary}'),
    ])
    | llm_instance
    | StrOutputParser()
)

# Compose: document → summary → (facts, tweet) in parallel
from langchain_core.runnables import RunnableParallel

full_pipeline = (
    {'summary': summarize_chain}               # step 1: summarize
    | RunnableParallel(
        summary = RunnablePassthrough(),        # pass summary through
        facts   = extract_chain,               # extract facts from summary
        tweet   = tweet_chain,                 # write tweet from summary
    )
)

print('Sequential + Parallel Chain Pipeline')
print('document → summarize → [extract facts | write tweet]')
print('=' * 60)

result = full_pipeline.invoke({'document': DOCUMENT})

print('Summary:')
print(' ', result['summary'])
print()
print('Extracted facts (JSON):')
print(' ', result['facts'][:200])
print()
print('Tweet:')
print(' ', result['tweet'])

In [ ]:
# ----------------------------------------------------------------
# Part 6 — Memory: Conversation state management
# ----------------------------------------------------------------

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# ConversationBufferMemory stores the full conversation history
# and automatically injects it into every prompt

memory = ConversationBufferMemory(
    return_messages = True,    # return as message objects
    memory_key      = 'history',
)

conversation = ConversationChain(
    llm     = get_llm(temperature=0.5),
    memory  = memory,
    verbose = False,
)

# Simulate a multi-turn conversation
turns = [
    'My name is Arjun and I am learning about RAG systems.',
    'What should I focus on next after understanding the retrieval part?',
    'Can you remind me what I told you my name was?',
]

print('ConversationChain with Buffer Memory')
print('=' * 60)

for i, user_input in enumerate(turns, 1):
    response = conversation.predict(input=user_input)
    print(f'Turn {i}')
    print(f'  User : {user_input}')
    print(f'  AI   : {response[:150]}...')
    print()

# Inspect the stored memory
print('Stored memory (last 2 message pairs):')
print('-' * 55)
for msg in memory.chat_memory.messages[-4:]:
    role = 'USER' if msg.type == 'human' else 'AI  '
    print(f'  [{role}] {msg.content[:80]}...')

In [ ]:
# ----------------------------------------------------------------
# Summary Memory — compresses old turns to save tokens
# Essential for long conversations that approach context limits
# ----------------------------------------------------------------

from langchain.memory import ConversationSummaryBufferMemory

# Keeps recent messages verbatim + a summary of older messages
# max_token_limit triggers summarization of oldest messages

summary_memory = ConversationSummaryBufferMemory(
    llm             = get_llm(temperature=0),
    max_token_limit = 200,      # summarize when history exceeds 200 tokens
    return_messages = True,
)

# Manually add some history to fill the buffer
summary_memory.save_context(
    {'input': 'What is the difference between BERT and GPT?'},
    {'output': 'BERT uses bidirectional attention for understanding tasks. '
               'GPT uses causal (left-to-right) attention for generation.'}
)
summary_memory.save_context(
    {'input': 'Which one is better for summarization?'},
    {'output': 'GPT-style models excel at generation tasks including summarization. '
               'BERT-style models are better for classification and NER.'}
)
summary_memory.save_context(
    {'input': 'What about question answering?'},
    {'output': 'Both can do QA. BERT extracts answer spans (extractive QA). '
               'GPT generates free-form answers (generative QA).'}
)

print('ConversationSummaryBufferMemory')
print('=' * 55)
print('Memory contents:')
print(summary_memory.load_memory_variables({})['history'])
print()
print('Key behavior:')
print('  When token count > max_token_limit, older messages')
print('  are replaced with a running summary.')
print('  Recent messages remain verbatim for full context.')

In [ ]:
# ----------------------------------------------------------------
# Part 7 — LangChain RAG Pipeline
# Rebuild Day 15 RAG using LangChain components
# Compare code complexity vs manual implementation
# ----------------------------------------------------------------

from langchain_community.vectorstores import FAISS as LangFAISS
from langchain_community.embeddings   import HuggingFaceEmbeddings
from langchain_core.documents         import Document as LCDocument
from langchain.text_splitter          import RecursiveCharacterTextSplitter
from langchain_core.prompts           import ChatPromptTemplate
from langchain_core.runnables         import RunnablePassthrough

# ---- Step 1: Create LangChain Documents ----
raw_texts = [
    ('Machine learning enables computers to learn from data. '
     'Supervised learning uses labeled examples. Unsupervised learning '
     'finds hidden structure. Overfitting occurs when a model memorizes '
     'training data and fails to generalize to new examples. '
     'Regularization and dropout are common remedies.',
     {'source': 'ml_basics.txt'}),

    ('The Transformer architecture uses self-attention to process '
     'sequences in parallel. BERT is encoder-only and bidirectional. '
     'GPT is decoder-only and autoregressive. '
     'Self-attention computes query, key, value vectors for each token '
     'and uses dot-product similarity to aggregate information.',
     {'source': 'transformers.txt'}),

    ('RAG combines retrieval and generation. Documents are chunked, '
     'embedded, and stored in a vector store. At query time, the most '
     'similar chunks are retrieved and injected into the LLM prompt. '
     'This grounds the answer in retrieved evidence and reduces hallucinations.',
     {'source': 'rag_intro.txt'}),
]

lc_docs = [LCDocument(page_content=text, metadata=meta)
           for text, meta in raw_texts]

# ---- Step 2: Chunk with LangChain splitter ----
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 300,
    chunk_overlap = 40,
)
chunks = splitter.split_documents(lc_docs)
print(f'Chunks created: {len(chunks)}')

# ---- Step 3: Embed + vector store ----
embeddings = HuggingFaceEmbeddings(
    model_name = 'all-MiniLM-L6-v2',
    model_kwargs = {'device': 'cpu'},
)
vectorstore = LangFAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 3})

print(f'Vector store ready with {vectorstore.index.ntotal} vectors')

# ---- Step 4: RAG chain using LCEL ----
rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Answer using ONLY the provided context. '
     'Cite the source filename after each claim. '
     'If the context does not contain the answer, say so.'),
    ('human',
     'Context:\n{context}\n\nQuestion: {question}'),
])

def format_docs(docs):
    return '\n\n'.join(
        f"[Source: {d.metadata['source']}]\n{d.page_content}"
        for d in docs
    )

lc_rag_chain = (
    {'context' : retriever | format_docs,
     'question': RunnablePassthrough()}
    | rag_prompt
    | get_llm(temperature=0.1)
    | StrOutputParser()
)

print()
print('LangChain RAG Chain Ready')
print('=' * 55)
print('Architecture: retriever | format_docs | rag_prompt | llm | parser')
print()

In [ ]:
# ---- Test the LangChain RAG chain ----

questions = [
    'What is overfitting and how do you prevent it?',
    'How does self-attention work?',
    'What are the benefits of RAG over pure LLM generation?',
    'What is the best pizza topping?',   # out-of-context → should refuse
]

print('LangChain RAG Results')
print('=' * 60)

for q in questions:
    answer = lc_rag_chain.invoke(q)
    print(f'Q: {q}')
    print(f'A: {answer[:200]}...')
    print()

In [ ]:
# ----------------------------------------------------------------
# Part 8 — OutputParsers: structured output from LLMs
# ----------------------------------------------------------------

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1   import BaseModel, Field
from typing import List

# Define the output schema using Pydantic
class ResearchSummary(BaseModel):
    title        : str         = Field(description='Paper title')
    key_finding  : str         = Field(description='Main result in 1 sentence')
    methods      : List[str]   = Field(description='List of methods used')
    limitations  : str         = Field(description='Main limitation')
    impact_score : int         = Field(description='Impact score 1-10')

parser = JsonOutputParser(pydantic_object=ResearchSummary)

extract_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extract structured information from research paper abstracts.'),
    ('human',  '{format_instructions}\n\nAbstract:\n{abstract}'),
]).partial(format_instructions=parser.get_format_instructions())

extract_chain = extract_prompt | get_llm(temperature=0) | parser

abstract = """
We present FlashAttention-3, a fast and memory-efficient exact attention
algorithm targeting H100 GPUs. By exploiting hardware features including
TMA and WGMMA instructions, we achieve 1.5-2.0x speedup over FlashAttention-2
reaching up to 740 TFLOPS on H100 SXM5. We evaluate on standard language
model benchmarks and show no degradation in model quality. The main
limitation is that the implementation is hardware-specific and does not
generalize beyond H100 GPUs without modification.
"""

print('Structured Output with JsonOutputParser + Pydantic')
print('=' * 60)

try:
    result = extract_chain.invoke({'abstract': abstract})
    import json
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f'Parser result (may vary by provider):')
    print('  title: FlashAttention-3')
    print('  key_finding: 1.5-2x speedup over FlashAttention-2')
    print('  methods: ["TMA instructions", "WGMMA", "H100 optimization"]')
    print('  limitations: Hardware-specific to H100 GPUs')
    print('  impact_score: 8')

In [ ]:
# ----------------------------------------------------------------
# Side-by-side comparison: manual RAG vs LangChain RAG
# ----------------------------------------------------------------

print('Manual RAG (Day 14-15) vs LangChain RAG (Day 16)')
print('=' * 65)

comparison = {
    'Document loading'   : ('Custom Document class + manual file reading',
                            'DirectoryLoader, PyPDFLoader, WebBaseLoader...'),
    'Text splitting'     : ('recursive_chunker() — 40 lines of code',
                            'RecursiveCharacterTextSplitter — 1 line'),
    'Embedding'          : ('SentenceTransformer.encode()',
                            'HuggingFaceEmbeddings, OpenAIEmbeddings...'),
    'Vector store'       : ('VectorStore class — 80 lines',
                            'FAISS.from_documents() — 1 line'),
    'Retriever'          : ('store.search() — manual',
                            '.as_retriever() — 1 line'),
    'Chain assembly'     : ('Manual prompt building + call_llm()',
                            'LCEL: retriever | prompt | llm | parser'),
    'Memory'             : ('ConversationManager class — 30 lines',
                            'ConversationBufferMemory — 2 lines'),
    'Tracing/observability': ('Manual logging',
                              'LangSmith integration (automatic)'),
    'When to use manual' : ('Learning, maximum control, custom logic',
                            '—'),
    'When to use LC'     : ('—',
                            'Production, teams, rapid iteration'),
}

for aspect, (manual, lc) in comparison.items():
    print(f'\n{aspect}:')
    print(f'  Manual    : {manual}')
    print(f'  LangChain : {lc}')

---

## Day 16 Summary

```
What you built today:

1.  get_llm()                →  unified provider wrapper (OpenAI/Anthropic/mock)
2.  PromptTemplate           →  parameterized prompt factories
3.  ChatPromptTemplate       →  system + user message structure
4.  LCEL pipes               →  prompt | llm | parser composition
5.  RunnableParallel         →  run multiple chains simultaneously
6.  ConversationBufferMemory →  full history injection
7.  ConversationSummaryBufferMemory → compress old turns
8.  LangChain RAG chain      →  retriever | format | prompt | llm | parser
9.  JsonOutputParser         →  structured Pydantic output extraction
10. Manual vs LangChain      →  when to use each

Key insight:
  LangChain is not magic — it is the same components you built
  manually, wrapped in composable abstractions.
  LCEL's pipe operator makes the data flow explicit and readable.

What comes next:
  Day 17 — Agents: giving LLMs tools to act in the world.
  You will build a ReAct agent that can search, calculate,
  and reason across multiple steps to answer complex questions.
```

### Self-Check Questions

1. What does the `|` operator do in LCEL?
2. When would you use `ConversationSummaryMemory` over `ConversationBufferMemory`?
3. What does `RunnablePassthrough()` do in a chain?
4. Why does the RAG chain pass `retriever | format_docs` together?
5. What is the difference between `.invoke()` and `.batch()`?